In [1]:
"""
ReAct Agent Example: Trip Cost Estimator
Pattern: Reasoning + Acting in a loop
 
The agent reasons about what it needs, decides to call tools (functions),
executes those tools, observes the results, and loops until it has an answer.
 
Structure:
  1. Configuration (LL  M setup)
  2. Tool definitions (functions the agent can call)
  3. Agent class (maintains conversation history)
  4. Action parsing (extract tool calls from LLM output)
  5. ReAct loop (the main agent loop)
  6. Usage examples
"""
import os
import re
from openai import OpenAI
from dotenv import load_dotenv
from openai import OpenAI
from typing import Optional, Any
from pydantic import BaseModel

In [2]:
# =============================================================================
# SECTION 1: CONFIGURATION
# =============================================================================
# Load environment variables and initialize the LLM client


class LLMSetup(BaseModel):
    llm_model: str = "gpt-4"
    llm_client: Any


def load_llm() -> LLMSetup:
    api_key = os.getenv("OPENAI_API_KEY")

    if not api_key:
        raise EnvironmentError("OPENAI_API_KEY not found in env")

    llm_client = OpenAI(api_key=api_key)
    return LLMSetup(llm_client=llm_client)

In [7]:
# =============================================================================
# SECTION 2: TOOL DEFINITIONS
# =============================================================================
# These are the functions the agent can call to gather information.
# The agent decides WHICH tool to call and WITH WHAT PARAMETERS.


def estimate_trip_cost(days: int, travelers: int, comfort: str = "mid") -> str:
    """
    Estimate a trip budget in SGD using simple heuristics.

    This is a TOOL that the agent can call. The agent provides:
    - days: number of days for the trip
    - travelers: number of people traveling
    - comfort: budget level (budget | mid | premium)

    Returns:
    - A string with the total estimated cost

    The agent uses this tool when it needs to know the cost of a trip.
    """
    # Validate inputs
    if days <= 0 or travelers <= 0:
        raise ValueError("days and travelers must be > 0")

    comfort = comfort.lower().strip()
    if comfort not in {"budget", "mid", "premium"}:
        raise ValueError("comfort must be one of: budget, mid, premium")

    # Cost estimates per person per day in SGD
    # These are rough heuristics, not real-world data
    daily_costs = {
        "budget": {
            "lodging": 60,
            "food": 30,
            "transport": 10,
            "activities": 20,
        },
        "mid": {
            "lodging": 140,
            "food": 60,
            "transport": 20,
            "activities": 50,
        },
        "premium": {
            "lodging": 300,
            "food": 120,
            "transport": 50,
            "activities": 120,
        },
    }

    # Calculate costs for all travelers over all days
    costs = daily_costs[comfort]
    lodging = costs["lodging"] * travelers * days
    food = costs["food"] * travelers * days
    transport = costs["transport"] * travelers * days
    activities = costs["activities"] * travelers * days

    subtotal = lodging + food + transport + activities
    # Add 12% buffer for contingencies
    contingency = round(subtotal * 0.12)
    total = subtotal + contingency

    # Return in a format the agent can parse
    return f"total_estimate: {total}"


# Dictionary of all available tools
# The agent can only call tools that are in this dictionary
AVAILABLE_TOOLS = {"estimate_trip_cost": estimate_trip_cost}

# =============================================================================
# SECTION 3: AGENT CLASS
# =============================================================================
# Maintains conversation history and communicates with the LLM


class ReActAgent:
    """
    A ReAct (Reasoning + Acting) agent that maintains conversation history
    and calls the LLM to reason about a problem and decide what actions to take.

    Key insight: The agent keeps ALL messages (system, user, assistant) in
    self.messages. This way, the LLM can see the entire conversation history
    and make decisions based on what has already been discussed.
    """

    def __init__(self, system_prompt: str, llm_setup: LLMSetup = load_llm()):
        """
        Initialize the agent with a system prompt.

        Args:
            system_prompt: Instructions for the agent (e.g., "You run in a
                          Thought-Action-Observation loop...")
        """
        self.llm = llm_setup
        self.messages = []
        if system_prompt:
            # Add system prompt as the first message
            # This tells the LLM how to behave
            self.messages.append({"role": "system", "content": system_prompt})

    def __call__(self, user_message: str) -> str:
        """
        Send a message to the agent and get a response.

        This method:
        1. Adds the user message to history
        2. Calls the LLM
        3. Adds the LLM response to history
        4. Returns the response

        Args:
            user_message: What the user or agent loop sends to the LLM

        Returns:
            The LLM's response (may contain Action, Thought, or Answer)
        """
        # Add user message to conversation history
        self.messages.append({"role": "user", "content": user_message})

        # Call the LLM with the full conversation history
        response = self._call_llm()

        # Add LLM response to conversation history
        # This is crucial: the next turn, the LLM will see this response
        self.messages.append({"role": "assistant", "content": response})

        return response

    def _call_llm(self) -> str:
        """
        Make an API call to the LLM with the current message history.

        The LLM sees all messages (system + entire conversation history),
        which allows it to understand context and make informed decisions.

        Returns:
            The LLM's response as a string
        """
        response = self.llm.llm_client.chat.completions.create(
            model=self.llm.llm_model,
            temperature=0.4,  # Lower temperature = more focused, deterministic
            messages=self.messages,
        )
        return response.choices[0].message.content


# =============================================================================
# SECTION 4: ACTION PARSING
# =============================================================================
# Extract what tool the agent wants to call and what parameters to use

# Regex to find lines like "Action: estimate_trip_cost: 2, 2, "mid""
ACTION_PATTERN = re.compile(r"^Action: (\w+): (.*)$")

# Regex to parse parameters for estimate_trip_cost
# Expected format: "2, 2, "mid"" or "2, 2, mid"
ESTIMATE_PARAMS_PATTERN = re.compile(
    r"^\s*(\d+)\s*,\s*(\d+)\s*,\s*\"?(budget|mid|premium)\"?\s*$", re.IGNORECASE
)


def parse_action_from_response(response_text: str) -> tuple:
    """
    Parse the agent's response to extract which tool it wants to call.

    The agent outputs text like:
      "I need to calculate trip costs.
       Action: estimate_trip_cost: 2, 2, mid"

    This function extracts:
    - action_name: "estimate_trip_cost"
    - action_input: "2, 2, mid"

    Args:
        response_text: The full response from the LLM

    Returns:
        tuple: (action_name, action_input) if found, otherwise (None, None)
    """
    # Split response into lines and look for Action lines
    for line in response_text.split("\n"):
        match = ACTION_PATTERN.match(line)
        if match:
            action_name, action_input = match.groups()
            return action_name, action_input

    # No action found (means the agent decided to return the Answer)
    return None, None


def execute_tool(tool_name: str, tool_input: str) -> str:
    """
    Execute a tool (function) with the given input.

    This function:
    1. Validates that the tool exists
    2. Parses the input parameters
    3. Calls the tool function
    4. Returns the result

    Args:
        tool_name: Name of the tool (e.g., "estimate_trip_cost")
        tool_input: Raw input string (e.g., "2, 2, mid")

    Returns:
        The result from the tool as a string

    Raises:
        ValueError: If the tool doesn't exist or parameters are invalid
    """
    if tool_name not in AVAILABLE_TOOLS:
        raise ValueError(f"Unknown tool: {tool_name}")

    # Tool-specific parsing and execution
    if tool_name == "estimate_trip_cost":
        # Parse the parameters: "days, travelers, comfort"
        match = ESTIMATE_PARAMS_PATTERN.match(tool_input)
        if not match:
            raise ValueError(f"Invalid parameters for estimate_trip_cost: {tool_input}")

        days = int(match.group(1))
        travelers = int(match.group(2))
        comfort = match.group(3).lower()

        print(f"  → Executing {tool_name}({days}, {travelers}, {comfort})")
        return AVAILABLE_TOOLS[tool_name](days, travelers, comfort)

    # Add other tool handling here as you add more tools
    raise ValueError(f"No execution handler for tool: {tool_name}")

In [8]:
# =============================================================================
# SECTION 5: REACT LOOP
# =============================================================================
# The main agent loop that reasons, acts, observes, and repeats


def run_agent_loop(question: str, max_turns: int = 10) -> None:
    """
    Run the ReAct agent loop for a question.

    Flow:
    1. Agent receives question
    2. Agent thinks and decides what action to take
    3. Agent calls a tool and gets an observation
    4. Agent feeds observation back and repeats
    5. Continue until agent returns an Answer (no more actions)

    Args:
        question: The initial question for the agent
        max_turns: Maximum number of loop iterations (safety limit)
    """
    print(f"\n{'=' * 70}")
    print(f"Question: {question}")
    print(f"{'=' * 70}\n")

    # Define how the agent should behave
    system_prompt = """
You run in a loop of Thought, Action, Observation.
At the end of the loop you output an Answer.
 
Use Thought to describe your thoughts about the question.
Use Action to run one of the actions available to you.
Observation will be the result of running those actions.
 
Important notes:
- Do not invent any facts or numbers. Use available actions instead.
- When you have enough information, provide your final Answer.
 
Your available actions are:
 
estimate_trip_cost:
  Format: Action: estimate_trip_cost: <days>, <travelers>, "<comfort>"
  Example: Action: estimate_trip_cost: 2, 2, "mid"
  Returns: the estimated cost of a trip
  comfort must be one of: budget, mid, premium
""".strip()

    # Create agent instance with the system prompt
    agent = ReActAgent(system_prompt=system_prompt)

    # Start the loop with the user's question
    next_prompt = question
    turn = 0

    while turn < max_turns:
        turn += 1
        print(f"--- Turn {turn} ---")

        # Call the agent
        agent_response = agent(next_prompt)
        print(f"Agent: {agent_response}\n")

        # Check if the agent wants to take an action
        action_name, action_input = parse_action_from_response(agent_response)

        if action_name:
            # The agent decided to use a tool
            print(f"→ Agent wants to call tool: {action_name}")

            try:
                # Execute the tool
                observation = execute_tool(action_name, action_input)
                print(f"  → Result: {observation}\n")

                # Feed the observation back to the agent
                # This becomes the input for the next turn
                next_prompt = f"Observation: {observation}"

            except ValueError as e:
                print(f"  ✗ Error: {e}\n")
                next_prompt = f"Error: {str(e)}. Please try a different action."

        else:
            # The agent returned an Answer (no more actions)
            print("✓ Agent has returned its final Answer. Stopping loop.\n")
            return

    print(f"⚠ Reached max turns ({max_turns}). Stopping loop.")


In [9]:
# Example 1: Simple query with one tool call
print("\n" + "=" * 70)
print("EXAMPLE 1: Simple Query")
print("=" * 70)
question_1 = (
    "A 2-day Tokyo trip for 2 adults. Mid comfort. How much roughly will it cost?"
)
run_agent_loop(question_1)


EXAMPLE 1: Simple Query

Question: A 2-day Tokyo trip for 2 adults. Mid comfort. How much roughly will it cost?

--- Turn 1 ---
Agent: Thought: The user wants to know the estimated cost of a 2-day trip to Tokyo for 2 adults with mid comfort. I can use the "estimate_trip_cost" action to find this information.

Action: estimate_trip_cost: 2, 2, "mid"

→ Agent wants to call tool: estimate_trip_cost
  → Executing estimate_trip_cost(2, 2, mid)
  → Result: total_estimate: 1210

--- Turn 2 ---
Agent: Answer: The estimated cost for a 2-day trip to Tokyo for 2 adults with mid-level comfort is approximately $1210.

✓ Agent has returned its final Answer. Stopping loop.

